In [ ]:
from experiment_management import (
    PromptRepository,
    ExperimentTemplateDTO,
    ExperimentTemplateRepository,
    ExperimentLiveInstanceRepository,
    ExperimentSnapshotRepository,
    GeneratedOutputRepository,
    init_schema
    )
import json

init_schema()
prompt_repo = PromptRepository()
et_repo = ExperimentTemplateRepository()
live_repo = ExperimentLiveInstanceRepository()
snap_repo = ExperimentSnapshotRepository()
gen_repo = GeneratedOutputRepository()


loss_kwargs = {"p": 2, "q": 2, "target_layer": -1}
loss_kwargs_json = json.dumps(loss_kwargs)
optimizer_kwargs = {"lr": 1e-3}
optimizer_kwargs_json = json.dumps(optimizer_kwargs)
group_id = 1
original_et_dto = et_repo.create_from_args(
    prompt_group=group_id,
    loss_name="melbo_lesswrong_loss",
    loss_additional_parameters=loss_kwargs_json,
    optimizer_name="AdamW",
    optimizer_additional_parameters=optimizer_kwargs_json,
    module_name="model.language_model.layers.6",
    batch_size=1,
    normalization=1.0,
)

et_dtos = et_repo.find_matching(original_et_dto,exclude=["normalization"],result_filter=None,orderby={"normalization":"asc"})
et_dtos.insert(0,original_et_dto)


In [ ]:
et_dto = et_dtos[0]
live_dtos = live_repo.get_all_from_experiment_template(et_dto, result_filter=None, orderby=None)
prompts = prompt_repo.get_prompts_from_group(group_id)
for prompt_index, prompt in enumerate(prompts):
    for live_index, live_dto in enumerate(live_dtos):
        snap_dto = snap_repo.get_all_from_live(
            live_dto,
            result_filter={"iteration_count": {"=": 0}},
            orderby=None,
        )[0]
        gen_dtos = gen_repo.get_all_from_snapshot(
            snap_dto,
            result_filter={"prompt_id": {"=": prompt.prompt_id}},
            orderby=None,
        )